In [11]:
import json
import re
import polars as pl
from pathlib import Path
from pydantic import BaseModel, Field
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

In [12]:

DATA_DIR = Path.cwd().parent.parent / "data"
CONTEXT_DIR = DATA_DIR / "context"


In [13]:
FEATURES = [
    "age", "gender", "side", "stone_localization", "stone_burden_cm2",
    "ct_scan", "comorbidity", "previous_surgery", "stone_anamnesis",
    "asa_score", "urine_culture", "creatinine", "solitary_kidney",
    "renal_anomaly", "additional_renal_disease",
]

COLUMN_MAP = {
    "YAŞ": "age",
    "CİNSİYET": "gender",
    "TARAF": "side",
    "LOKALİZASYON": "stone_localization",
    "TOPLAM TAŞ YÜKÜ (CM2)": "stone_burden_cm2",
    "BT": "ct_scan",
    "ÖZGEÇMİŞ": "comorbidity",
    "GEÇİRİLMİŞ CERRAHİ": "previous_surgery",
    "TAS ANAMNEZI": "stone_anamnesis",
    "ASA SKORU": "asa_score",
    "İKAB": "urine_culture",
    "KREATİNİN": "creatinine",
    "SOLİTER BB": "solitary_kidney",
    "RENAL ANOMALİ": "renal_anomaly",
    "EK RENAL HASTALIK": "additional_renal_disease",
    "SONUÇ-2": "result",
}


In [14]:

class PredictionOutput(BaseModel):
    prediction: int = Field(description="1 for success (stone-free), 2 for failure (residual fragments)")
    reasoning: str = Field(description="A single short sentence explaining the prediction")
    confidence: str = Field(description="Model confidence: 'low', 'medium', or 'high'")



In [15]:
df = pl.read_csv(DATA_DIR / "tabular" / "pediatric-pcnl.csv", infer_schema_length=None)
df = df.rename(COLUMN_MAP).select(FEATURES + ["result"])
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
display(df.head(3))


Loaded 269 rows, 16 columns


age,gender,side,stone_localization,stone_burden_cm2,ct_scan,comorbidity,previous_surgery,stone_anamnesis,asa_score,urine_culture,creatinine,solitary_kidney,renal_anomaly,additional_renal_disease,result
f64,i64,i64,str,str,i64,i64,f64,i64,i64,str,str,i64,str,i64,i64
13.0,2,1,"""5""","""15""",1,0,4.0,0,1,"""0""","""0.7""",0,"""0""",0,1
8.0,1,1,"""4""","""2,50""",0,0,0.0,0,1,"""0""","""0.5""",0,"""0""",0,1
3.0,1,1,"""20""","""1,50""",0,0,0.0,0,1,"""0""","""0.4""",0,"""0""",0,1
